# Phase 10 — Adaptive Multi-Expert Composition
## NeuroForge Experimental Research Framework

### Core Research Question
Can allowing more than one heterogeneous expert to execute on an individual sample improve performance on genuinely mixed-structure tasks, while preserving the adaptive-computation advantage of using fewer experts on samples that do not require them?

> **Mandatory Disclaimer:** The single-expert limits identified in Phase 9 were empirical ceilings for the evaluated specialist portfolio and task construction, not universal mathematical limits for all conceivable neural architectures. On composite mixed-structure samples where no single specialist architecture solves the combined task, the oracle is strictly designated as **ORACLE NOT DEFINED**.

## 1. Environment Validation
Verify Python, PyTorch, device, and runtime environment.

In [ ]:
import platform
import sys
from pathlib import Path
import torch
import neuroforge

print(f"Python Version:  {platform.python_version()} ({platform.architecture()[0]})")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
print(f"Default Device:  {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"NeuroForge Path: {neuroforge.__file__}")

## 2. Existing Baseline Verification
In Phase 9, single-expert learned routing ($k=1$) achieved 93.3% on pure families, but encountered an empirical single-expert ceiling of ~75% on composite mixed-structure tasks (FR, RC, FC, FRC) due to discordant sub-signals.

In [ ]:
import json

phase9_summary_path = Path("../results/metrics/phase9_generalization/phase9_summary.json")
if phase9_summary_path.exists():
    with phase9_summary_path.open("r", encoding="utf-8") as f:
        p9_data = json.load(f)
    print("Phase 9 Empirical Ceilings on Mixed Tasks:")
    print(f"  Single-Expert Router Accuracy: {p9_data.get('regime_9d_mixed', {}).get('router_accuracy', 0.743)*100:.2f}%")
    print(f"  Single-Expert Ceiling:         {p9_data.get('regime_9d_mixed', {}).get('ceiling_accuracy', 0.750)*100:.2f}%")
    print(f"  Primary Failure Diagnosis:     {p9_data.get('regime_9d_mixed', {}).get('primary_failure', 'Failure D — Single-Expert Ceiling')}")
else:
    print("Phase 9 summary: Baseline single-expert router accuracy ~74.3%, ceiling ~75.0%.")

## 3. Phase 10 Hypothesis Registration
- **H1 (Multi-Expert Composition Advantage):** On genuinely mixed-structure tasks, fixed $k=2$ and fixed $k=3$ parallel execution will surpass the Phase 9 single-expert ceiling (>80%, approaching 90%+).
- **H2 (Adaptive-k Selectivity):** A learned adaptive-k router ($k \in \{1, 2, 3\}$) will allocate $k \approx 1$ to pure single-family inputs and $k \ge 2$ to composite inputs without task-family metadata.
- **H3 (Structural Composition Specificity):** Multi-expert composition advantage will be concentrated on mixed-structure tasks; on pure tasks, $k > 1$ yields diminishing returns while multiplying compute.
- **H4 (Compute-Aware Frontier):** Penalizing compute via $\lambda$ enables controllable operating points between $k=1$ compute and $k=3$ peak accuracy.
- **H5 (Mechanism Superiority):** Parallel logit aggregation will outperform sequential specialist chaining due to latent representation misalignment between frozen specialists.

## 4. Dataset Verification
Inspecting the 7 synthetic task families: 3 pure (F, R, C) and 4 composite mixed structures (FR, RC, FC, FRC).

In [ ]:
from collections import Counter
from neuroforge.datasets.phase9_datasets import Phase9MixedStructureDataset

dataset = Phase9MixedStructureDataset(samples_per_type=20, seed=42)
print(f"Total dataset samples: {len(dataset)}")
print(f"Sample features shape: {dataset[0]['features'].shape}")

family_counts = Counter(dataset[i]['family'] for i in range(len(dataset)))
print("Sample distribution across 7 task families:")
for fam, count in sorted(family_counts.items()):
    print(f"  {fam:>5}: {count} samples")

## 5. Specialist Verification
Inspecting capacity-matched frozen specialists: MLP (depth=3), Graph (depth=2), Attention (depth=1), AttentionBlockV2 (depth=3).

In [ ]:
from neuroforge.models.specialists import StandaloneSpecialist

specialists = {
    "mlp": StandaloneSpecialist("mlp", input_dim=8, hidden_dim=24, depth=3),
    "graph": StandaloneSpecialist("graph", input_dim=8, hidden_dim=24, depth=2),
    "attention": StandaloneSpecialist("attention", input_dim=8, hidden_dim=24, depth=1),
    "attention_v2": StandaloneSpecialist("attention_v2", input_dim=8, hidden_dim=24, depth=3),
}
for name, spec in specialists.items():
    spec.eval()
    for p in spec.parameters():
        p.requires_grad = False
    p_count = sum(p.numel() for p in spec.parameters())
    print(f"Specialist '{name:<12}': depth={spec.depth}, params={p_count}")

## 6. Fixed k=2 Routing Evaluation
Evaluating TopKRouter with $k=2$ across the benchmark.

In [ ]:
from neuroforge.routing.multi_expert import TopKRouter
from neuroforge.models.composition import ParallelMultiExpert

router_k2 = TopKRouter(input_dim=8, hidden_dim=16, num_experts=4)
pme_k2 = ParallelMultiExpert(specialists, router=router_k2, aggregation="uniform")

sample_x = dataset[0]["features"].unsqueeze(0)
logits_k2, dec_k2, flops_k2 = pme_k2(sample_x, k=2)
print("Top-K Router (k=2) Decision:")
print(f"  Selected indices: {dec_k2.selected_indices[0]}")
print(f"  Weights:          {dec_k2.weights[0].tolist()}")
print(f"  Sample FLOPs:     {flops_k2[0]:.1f}")

## 7. Fixed k=3 Routing Evaluation
Evaluating TopKRouter with $k=3$, activating three heterogeneous specialists simultaneously.

In [ ]:
logits_k3, dec_k3, flops_k3 = pme_k2(sample_x, k=3)
print("Top-K Router (k=3) Decision:")
print(f"  Selected indices: {dec_k3.selected_indices[0]}")
print(f"  Weights:          {dec_k3.weights[0].tolist()}")
print(f"  Sample FLOPs:     {flops_k3[0]:.1f}")

## 8. Parallel Aggregation Mechanisms
Comparing C1 (Uniform Averaging), C2 (Router-Weighted Averaging), and C3 (Normalized Probability Combination).

In [ ]:
from neuroforge.routing.multi_expert import (
    aggregate_parallel_uniform,
    aggregate_parallel_weighted,
    aggregate_parallel_normalized,
)

z_mlp = torch.tensor([[2.5, -1.0]])
z_graph = torch.tensor([[-0.5, 3.2]])
mock_logits = [z_mlp, z_graph]
mock_weights = torch.tensor([[0.65, 0.35]])

c1 = aggregate_parallel_uniform(mock_logits)
c2 = aggregate_parallel_weighted(mock_logits, mock_weights)
c3 = aggregate_parallel_normalized(mock_logits, mock_weights)

print("Parallel Aggregation Comparison:")
print(f"  C1 (Uniform):    {c1.tolist()}")
print(f"  C2 (Weighted):   {c2.tolist()}")
print(f"  C3 (Normalized): {c3.tolist()}")

## 9. Sequential Composition Pipelines
Comparing parallel aggregation against sequential specialist pipelines (e.g. Graph -> MLP, MLP -> Graph, AttnV2 -> MLP).

In [ ]:
from neuroforge.models.composition import SequentialSpecialistComposition

seq_pipe = SequentialSpecialistComposition(specialists, ("graph", "mlp"), mode="block_chain")
seq_out, seq_fl = seq_pipe(sample_x)
print(f"Sequential Graph->MLP Output Shape: {seq_out.shape}, FLOPs: {seq_fl:.1f}")

## 10. Adaptive k Routing
Evaluating dynamic sample-level expert allocation ($k \in \{1, 2, 3\}$) using learned k-head, entropy, and margin heuristics.

In [ ]:
from neuroforge.routing.multi_expert import AdaptiveKRouter

adapt_router = AdaptiveKRouter(input_dim=8, hidden_dim=16, num_experts=4)
dec_learned = adapt_router(sample_x, strategy="learned")
print("Adaptive-k Router Decision:")
print(f"  Chosen k:          {dec_learned.k_values[0].item()}")
print(f"  Selected experts:  {dec_learned.selected_indices[0]}")
print(f"  k-head logits:     {dec_learned.k_logits[0].tolist()}")
print(f"  Entropy:           {dec_learned.entropy[0].item():.4f}")

## 11. Compute-Aware Adaptive Composition
Evaluating trade-offs across cost penalty hyperparameter $\lambda \in \{0.0, 0.01, 0.10, 0.30, 1.0\}$.

In [ ]:
import pandas as pd

metrics_dir = Path("../results/metrics/phase10_multi_expert")
comp_csv = metrics_dir / "compute_results.csv"
if comp_csv.exists():
    df_comp = pd.read_csv(comp_csv)
    print(df_comp.to_string(index=False))
else:
    print("Results CSV available after running scripts/phase10_multi_expert.py")

## 12. Controlled Ablations
Ablating router representations, aggregation modes, and temperature parameter.

In [ ]:
abl_csv = metrics_dir / "ablation_results.csv"
if abl_csv.exists():
    df_abl = pd.read_csv(abl_csv)
    print(df_abl.to_string(index=False))
else:
    print("Ablation CSV available after running scripts/phase10_multi_expert.py")

## 13. Failure Localization & Diagnostic Audits
Auditing pre-declared failure modes (A through F) and collapse conditions.

In [ ]:
fail_csv = metrics_dir / "failure_diagnosis.csv"
if fail_csv.exists():
    df_fail = pd.read_csv(fail_csv)
    cols = ["family", "k1_accuracy", "multi_accuracy", "ceiling_accuracy", "failure_mode"]
    print(df_fail[[c for c in cols if c in df_fail.columns]].to_string(index=False))
else:
    print("Failure diagnosis CSV available after running scripts/phase10_multi_expert.py")

## 14. Pareto Analysis & Performance-Compute Frontiers
Visualizing non-dominated frontiers across compute budgets and target accuracy constraints.

In [ ]:
from IPython.display import Image, display

fig_path = Path("../figures/phase10_multi_expert/pareto_performance_compute.png")
if fig_path.exists():
    display(Image(filename=str(fig_path)))
else:
    print(f"Figure will be generated at {fig_path}")

## 15. Final Scientific Verdict

Verdict computed from the experiment artifacts below; no hypothesis is asserted without data support.


In [ ]:
import json
from pathlib import Path

summary_path = Path("../results/metrics/phase10_multi_expert/summary.json")
if not summary_path.exists():
    print("summary.json not found. Run scripts/phase10_multi_expert.py first "
          "to generate the experiment artifacts, then re-run this cell.")
else:
    with summary_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    sp = data.get("summary_by_policy", {})
    cvm = data.get("ceiling_vs_multi", {})
    collapse = data.get("collapse_audit", {})

    def pol(name):
        for key, v in sp.items():
            if key == name:
                return v
        return None

    def pct(x):
        return None if x is None else 100.0 * x

    k1 = pol("k=1 Baseline (Phase 8B)")
    fixed_k2 = pol("Fixed k=2 (Uniform C1)")
    fixed_k3 = pol("Fixed k=3")
    seq_attn_mlp = pol("Sequential AttnV2->MLP")

    def g(policy, key, default=None):
        return policy.get(key, default) if policy else default

    k1_mixed = g(k1, "mixed_accuracy_mean")
    k2_mixed = g(fixed_k2, "mixed_accuracy_mean")
    k3_mixed = g(fixed_k3, "mixed_accuracy_mean")
    k1_pure = g(k1, "pure_accuracy_mean")
    k3_pure = g(fixed_k3, "pure_accuracy_mean")

    def best_k(v):
        return max((v.get(kk) for kk in ("k1", "k2", "k3") if v.get(kk) is not None), default=None)

    print("=== Data-Grounded Hypothesis Verdicts (Phase 10) ===")

    # H1: fixed k>1 beats k=1 baseline on mixed-structure tasks, ideally breaking the ceiling
    if None in (k1_mixed, k2_mixed, k3_mixed):
        print("H1 [INCONCLUSIVE]: missing mixed accuracy means for k=1/k=2/k=3.")
    else:
        h1_beats = (k2_mixed > k1_mixed) or (k3_mixed > k1_mixed)
        ceiling_exceeded = any((best_k(v) is not None and best_k(v) > v.get("ceiling", float("inf")))
                               for v in cvm.values())
        verb = "SUPPORTED" if (h1_beats and ceiling_exceeded) else ("PARTIAL" if h1_beats else "NOT SUPPORTED")
        print(f"H1 (composition beats k=1 baseline on mixed): {verb} "
              f"[k=1 {pct(k1_mixed):.1f}% -> k=2 {pct(k2_mixed):.1f}% / k=3 {pct(k3_mixed):.1f}% mixed]; "
              f"exceeds single-expert ceiling on any mixed family: {ceiling_exceeded}")

    # H2: adaptive k allocates k~1 to pure, k>=2 to composite
    kd = data.get("k_distribution", {})
    pure = [f for f in ("F", "R", "C") if f in kd]
    mixed = [f for f in ("FR", "RC", "FC", "FRC") if f in kd]
    pure_k1 = (sum(kd[f].get("p_k1", 0) for f in pure) / len(pure)) if pure else None
    mixed_k2p = (sum(kd[f].get("p_k2", 0) + kd[f].get("p_k3", 0) for f in mixed) / len(mixed)) if mixed else None
    if pure_k1 is None or mixed_k2p is None:
        print("H2 [INCONCLUSIVE]: insufficient k_distribution data.")
    else:
        h2 = (pure_k1 >= 0.5) and (mixed_k2p >= 0.5)
        print(f"H2 (k~1 pure / k>=2 composite allocation): {'SUPPORTED' if h2 else 'NOT SUPPORTED'} "
              f"[mean p(k=1) on pure {100*pure_k1:.0f}%, mean p(k>=2) on mixed {100*mixed_k2p:.0f}%]")

    # H3: composition advantage concentrated on mixed; pure gains ~0
    if None in (k1_pure, k3_pure):
        print("H3 [INCONCLUSIVE]: missing pure accuracy means.")
    else:
        pure_gain = k3_pure - k1_pure
        mixed_gain = k3_mixed - k1_mixed
        h3 = (mixed_gain > pure_gain) and (pure_gain < 0.03)
        print(f"H3 (advantage concentrated on mixed, pure gains ~0): {'SUPPORTED' if h3 else 'NOT SUPPORTED'} "
              f"[pure k1->k3 gain {100*pure_gain:+.1f}pp, mixed k1->k3 gain {100*mixed_gain:+.1f}pp]")

    # H4: cost penalty lambda yields controllable operating points
    l_policies = ["Adaptive k (Cost-Aware λ=0.01)", "Adaptive k (Cost-Aware λ=0.10)",
                  "Adaptive k (Cost-Aware λ=0.30)", "Adaptive k (Cost-Aware λ=1.00)"]
    flops_set = {round(g(pol(p), "mean_flops"), 0) for p in l_policies if pol(p)}
    if len(flops_set) < 2:
        print("H4 [INCONCLUSIVE]: insufficient cost-aware operating points.")
    else:
        print(f"H4 (controllable lambda operating points): SUPPORTED "
              f"[distinct mean FLOPs across lambda sweep: {sorted(flops_set)}]")

    # H5: parallel aggregation beats sequential on mixed tasks
    seq_mixed = g(seq_attn_mlp, "mixed_accuracy_mean")
    if None in (k2_mixed, seq_mixed):
        print("H5 [INCONCLUSIVE]: missing parallel/sequential mixed means.")
    else:
        h5 = k2_mixed > seq_mixed
        print(f"H5 (parallel > sequential on mixed): {'SUPPORTED' if h5 else 'NOT SUPPORTED'} "
              f"[parallel k=2 mixed {pct(k2_mixed):.1f}% vs best sequential (AttnV2->MLP) mixed {pct(seq_mixed):.1f}%]")

    print()
    print("=== Overall CASE Verdict (Section 36 categories) ===")
    # Decision tree (in order): no composition exceeds ceiling -> F;
    # fixed-k exceeds but adaptive does not -> C; accuracy up but FLOPs blow up -> B; else A.
    ceiling_exceeded_any = any((best_k(v) is not None and best_k(v) > v.get("ceiling", float("inf")))
                               for v in cvm.values())
    fixed_exceeds = any((best_k(v) is not None and best_k(v) > v.get("ceiling", float("inf")))
                        and any(v.get(kk) is not None and v.get(kk) > v.get("ceiling", float("inf"))
                                for kk in ("k2", "k3"))
                        for v in cvm.values())
    adaptive_policies = ["Adaptive k (Cost-Aware λ=0.01)", "Adaptive k (Cost-Aware λ=0.10)",
                         "Adaptive k (Cost-Aware λ=0.30)", "Adaptive k (Cost-Aware λ=1.00)",
                         "Adaptive k (Entropy-Based)", "Adaptive k (Learned)", "Adaptive k (Margin-Based)"]
    adaptive_mixed = max((g(pol(p), "mixed_accuracy_mean", 0) for p in adaptive_policies), default=0)
    max_ceiling = max((v.get("ceiling", 0) for v in cvm.values()), default=0)

    k3_flops = g(fixed_k3, "mean_flops", 0) or 0
    k1_flops = g(k1, "mean_flops", 0) or 0
    flop_ratio = (k3_flops / k1_flops) if k1_flops else None

    if not cvm:
        print("Overall [INCONCLUSIVE]: ceiling_vs_multi absent; cannot compute CASE.")
    elif not ceiling_exceeded_any:
        print("Overall CASE: F - no composition exceeds the single-expert ceiling on any mixed family "
              "(multi-expert does not break the Phase 9 ceiling).")
    elif fixed_exceeds and adaptive_mixed <= max_ceiling:
        print("Overall CASE: C - fixed-k composition exceeds the ceiling but adaptive routing does not.")
    elif (k3_mixed is not None and k1_mixed is not None and k3_mixed > k1_mixed) and (flop_ratio and flop_ratio > 2.0):
        print(f"Overall CASE: B - accuracy improves (mixed {pct(k1_mixed):.1f}%->{pct(k3_mixed):.1f}%) "
              f"but FLOPs multiply x{flop_ratio:.1f}.")
    else:
        print("Overall CASE: A (Strong Composition Success) - multi-expert exceeds the single-expert ceiling.")

    print(f"  max single-expert ceiling across mixed families: {100*max_ceiling:.1f}%")
    print(f"  best adaptive-k mixed accuracy: {100*adaptive_mixed:.1f}%")
    if flop_ratio:
        print(f"  FLOPs multiple (k=3 / k=1): x{flop_ratio:.2f}")
    print(f"  collapse_audit: {collapse}")
